# Authenfy T4 validation and specialist export
Select a free T4 runtime. Run all cells in order. This notebook performs inference/export, not training.
The labeled example audit is correlated and is not a production benchmark.

In [ ]:
"""Run in a Colab T4 cell. Downloads a JSON receipt; never trains on smoke inputs."""
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.4.0", "onnxruntime==1.31.0"], check=True)
import hashlib
import io
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import onnxruntime as ort
import requests
import torch
from huggingface_hub import hf_hub_download
from PIL import Image, ImageOps
from transformers import ViTForImageClassification
from transformers.models.clip.image_processing_pil_clip import CLIPImageProcessorPil

MODEL_ID = "buildborderless/CommunityForensics-DeepfakeDet-ViT"
REVISION = "ac6ee457bea904a373065754107451793b56db00"
EXPECTED = "a42c7d740fbb345ba9a26d469b22f301d73089ce3c6da993877ed2b6965a8ba1"
assert torch.cuda.is_available(), "A CUDA GPU is required. Select Runtime > Change runtime type > T4 GPU."
gpu = torch.cuda.get_device_name(0)
assert "T4" in gpu, f"Expected T4, got {gpu}"
print("GPU:", gpu, flush=True)
model = ViTForImageClassification.from_pretrained(MODEL_ID, revision=REVISION, use_safetensors=True).eval().cuda()
processor = CLIPImageProcessorPil.from_pretrained(MODEL_ID, revision=REVISION)
path = hf_hub_download(MODEL_ID, "onnx/model.onnx", revision=REVISION)
digest = hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert digest == EXPECTED
options = ort.SessionOptions()
options.intra_op_num_threads = 2
session = ort.InferenceSession(path, sess_options=options, providers=["CPUExecutionProvider"])


def prepare(image):
    image = ImageOps.exif_transpose(image).convert("RGB")
    w, h = image.size
    scale = 440 / min(w, h)
    image = image.resize((int(w * scale), int(h * scale)), Image.Resampling.BICUBIC)
    left, top = (image.width - 384) // 2, (image.height - 384) // 2
    image = image.crop((left, top, left + 384, top + 384))
    mean = np.array([0.48145466, 0.4578275, 0.40821073], np.float32)
    std = np.array([0.26862954, 0.26130258, 0.27577711], np.float32)
    return np.ascontiguousarray(((np.asarray(image, np.float32) / 255 - mean) / std).transpose(2, 0, 1)[None])


rng = np.random.default_rng(42)
inputs = [("gray", Image.new("RGB", (384, 384), (128, 128, 128))),
          ("rectangular_noise", Image.fromarray(rng.integers(0, 256, (320, 640, 3), dtype=np.uint8))),
          ("portrait_noise", Image.fromarray(rng.integers(0, 256, (640, 320, 3), dtype=np.uint8)))]
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
r = requests.get(url, timeout=60)
r.raise_for_status()
inputs.append(("public_cat_photo", Image.open(io.BytesIO(r.content)).convert("RGB")))
records = []
for name, image in inputs:
    x = prepare(image)
    reference = processor(image, return_tensors="np")["pixel_values"]
    preprocessing_error = float(np.max(np.abs(x - reference)))
    assert preprocessing_error < 1e-5, (name, "preprocess mismatch", preprocessing_error)
    tensor = torch.from_numpy(x).cuda()
    with torch.inference_mode():
        model(pixel_values=tensor)
        torch.cuda.synchronize()
        start = time.perf_counter()
        logit = model(pixel_values=tensor).logits
        torch.cuda.synchronize()
        gpu_ms = (time.perf_counter() - start) * 1000
        p_gpu = torch.sigmoid(logit).item()
    start = time.perf_counter()
    raw = float(session.run(None, {"pixel_values": x})[0].reshape(-1)[0])
    cpu_ms = (time.perf_counter() - start) * 1000
    p_cpu = float(1 / (1 + np.exp(-np.clip(raw, -80, 80))))
    row = {"input": name, "rgb_sha256": hashlib.sha256(image.tobytes()).hexdigest(),
           "gpu_score": p_gpu, "onnx_score": p_cpu, "absolute_error": abs(p_gpu-p_cpu),
           "preprocessing_max_error": preprocessing_error, "gpu_ms": gpu_ms, "cpu_ms": cpu_ms}
    records.append(row)
    print(json.dumps(row), flush=True)
    assert row["absolute_error"] < 1e-4, "GPU/ONNX parity failed"
report = {"status": "passed", "kind": "runtime_smoke_and_numerical_parity", "gpu": gpu,
          "utc": datetime.now(timezone.utc).isoformat(), "torch": torch.__version__,
          "onnxruntime": ort.__version__, "preprocessor": "CLIPImageProcessorPil", "model": MODEL_ID, "revision": REVISION,
          "model_sha256": digest, "inputs": records,
          "limitations": "Four smoke inputs; no detection accuracy, calibration, face-swap or audio validation."}
Path("authenfy-colab-validation.json").write_text(json.dumps(report, indent=2))
print("AUTHENFY_T4_VALIDATION_PASSED", json.dumps(report), flush=True)
from google.colab import files
files.download("authenfy-colab-validation.json")


In [ ]:
"""Small labeled face-swap audit after colab_validate.py, not a benchmark."""
from concurrent.futures import ThreadPoolExecutor

REPO = "yermandy/deepfake-detection"
COMMIT = "0b872f7903e72d2af8c5fff262d2fc2ca99bb44f"
tree_response = requests.get(f"https://api.github.com/repos/{REPO}/git/trees/{COMMIT}?recursive=1", timeout=60)
tree_response.raise_for_status()
paths = sorted(row["path"] for row in tree_response.json()["tree"] if row["path"].endswith(".png") and row["path"].startswith("datasets/"))


def fetch(path):
    response = requests.get(f"https://raw.githubusercontent.com/{REPO}/{COMMIT}/{path}", timeout=60)
    response.raise_for_status()
    image = Image.open(io.BytesIO(response.content)).convert("RGB")
    label = 0 if "/real/" in path or "/Celeb-real/" in path or "/YouTube-real/" in path else 1
    return path, image, label, hashlib.sha256(response.content).hexdigest()


with ThreadPoolExecutor(max_workers=4) as pool:
    samples = list(pool.map(fetch, paths))
face_records = []
for path, image, label, digest in samples:
    x = torch.from_numpy(prepare(image)).cuda()
    with torch.inference_mode():
        score = torch.sigmoid(model(pixel_values=x).logits).item()
    face_records.append({"path": path, "label_fake": label, "score": score, "sha256": digest,
                         "group": "/".join(path.split("/")[:-1])})
labels = np.array([row["label_fake"] for row in face_records])
scores = np.array([row["score"] for row in face_records])
assert len(face_records) == 32 and set(labels) == {0, 1}
tp = int(((scores >= .5) & (labels == 1)).sum())
tn = int(((scores < .5) & (labels == 0)).sum())
fp = int(((scores >= .5) & (labels == 0)).sum())
fn = int(((scores < .5) & (labels == 1)).sum())
face_report = {"kind": "tiny_labeled_face_swap_gap_audit", "gpu": gpu, "utc": datetime.now(timezone.utc).isoformat(),
               "model": MODEL_ID, "model_revision": REVISION, "sample_source": REPO, "sample_commit": COMMIT,
               "n_images": len(face_records), "n_groups": len(set(row["group"] for row in face_records)),
               "fake_count": int(labels.sum()), "real_count": int((labels == 0).sum()),
               "accuracy_at_0_5": (tp+tn)/len(labels), "tp": tp, "tn": tn, "fp": fp, "fn": fn,
               "false_low_signal_fake_images": int(((scores <= .2) & (labels == 1)).sum()),
               "high_signal_fake_images": int(((scores >= .8) & (labels == 1)).sum()),
               "limitations": "32 correlated example crops from 8 video/method groups; unknown training overlap; not an independent test, calibration set, or production accuracy estimate. No model fitting or threshold tuning.",
               "records": face_records}
Path("authenfy-face-audit.json").write_text(json.dumps(face_report, indent=2))
print("AUTHENFY_FACE_AUDIT", json.dumps({key:value for key,value in face_report.items() if key != "records"}), flush=True)
files.download("authenfy-face-audit.json")


In [ ]:
"""Inspect the pinned published facial-manipulation model's tensor structure."""
FACE_MODEL_ID = "yermandy/deepfake-detection"
FACE_REVISION = "9a6857ec642deb57373c5437be803a199468b8c6"
FACE_SHA = "55c54d3f4d301ce07f0d08126fc4202572d20ea6952b039bc7a949a43714157d"
face_path = hf_hub_download(FACE_MODEL_ID, "model.torchscript", revision=FACE_REVISION)
assert hashlib.sha256(Path(face_path).read_bytes()).hexdigest() == FACE_SHA
face_archive = torch.jit.load(face_path, map_location="cpu")
face_state = face_archive.state_dict()
print("FACE_TENSOR_STRUCTURE", json.dumps({key:list(value.shape) for key,value in face_state.items() if "linear" in key or "embeddings" in key or "layer_norm" in key or "ln_" in key})[:6000], flush=True)
print("FACE_FIRST_KEYS", list(face_state)[:20], flush=True)
print("FACE_GRAPH", str(face_archive.code)[:2000], flush=True)


In [ ]:
"""Reconstruct pinned tensor weights using registry-installed Transformers; export FP32 ONNX."""
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx==1.20.1"], check=True)
from transformers import CLIPVisionConfig, CLIPVisionModel
import torch.nn.functional as F

print("FACE_HEAD_GRAPH", face_archive.model.code, flush=True)
config = CLIPVisionConfig(hidden_size=1024, intermediate_size=4096, num_hidden_layers=24,
                         num_attention_heads=16, image_size=224, patch_size=14,
                         hidden_act="quick_gelu", layer_norm_eps=1e-5, attention_dropout=0.0)
config._attn_implementation = "eager"
vision = CLIPVisionModel(config)
converted = {}
prefix = "feature_extractor.base_model.model.vision_model."
for key, value in face_state.items():
    if not key.startswith(prefix) or key.endswith("position_ids") or ".base_layer." in key:
        continue
    name = key[len(prefix):].replace(".ln_tuning_layers.default.", ".")
    converted[name] = value.float()
vision.vision_model.load_state_dict(converted, strict=True)


class FaceClassifier(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.vision = vision
        self.linear = torch.nn.Linear(1024, 2)
        self.linear.weight.data.copy_(face_state["model.linear.weight"].float())
        self.linear.bias.data.copy_(face_state["model.linear.bias"].float())

    def forward(self, pixel_values):
        features = self.vision(pixel_values=pixel_values).pooler_output
        return self.linear(F.normalize(features, p=2, dim=1))


def prepare_face(image):
    image = image.convert("RGB")
    w, h = image.size
    scale = 224 / min(w, h)
    image = image.resize((int(w * scale), int(h * scale)), Image.Resampling.BICUBIC)
    left, top = (image.width - 224) // 2, (image.height - 224) // 2
    image = image.crop((left, top, left + 224, top + 224))
    mean = np.array([0.48145466, .4578275, .40821073], np.float32)
    std = np.array([.26862954, .26130258, .27577711], np.float32)
    return np.ascontiguousarray(((np.asarray(image, np.float32) / 255 - mean) / std).transpose(2, 0, 1)[None])


face_model = FaceClassifier().eval().cuda().float()
# The published trace hard-codes BF16 casts. Replay it in FP32 for an apples-to-apples
# architecture check, rather than comparing different numerical precision paths.
face_archive = torch.jit.load(face_path, map_location="cpu").eval().cuda().float()
for module in face_archive.modules():
    if module._c._has_method("forward"):
        graph = module.graph
        for node in list(graph.nodes()):
            if node.kind() == "aten::to":
                for index, value in enumerate(list(node.inputs())[1:3], start=1):
                    if value.node().kind() == "prim::Constant" and value.toIValue() == 15:
                        replacement = graph.insertConstant(6)
                        replacement.node().moveBefore(node)
                        node.replaceInput(index, replacement)
specialist_records = []
with torch.inference_mode():
    for path, image, label, digest in samples:
        x = torch.from_numpy(prepare_face(image)).cuda()
        expected = face_archive(x).softmax(dim=1)[0, 1].item()
        actual = face_model(x).softmax(dim=1)[0, 1].item()
        assert abs(expected-actual) < 1e-4, (path, expected, actual, "reconstruction mismatch")
        specialist_records.append({"path":path, "label_fake":label, "sha256":digest,
                                   "published_graph_fp32_score":expected, "reconstructed_score":actual})
print("FACE_RECONSTRUCTION_PASSED", flush=True)
del face_archive, face_state, converted
import gc
gc.collect()
torch.cuda.empty_cache()
dummy = torch.zeros(1, 3, 224, 224, device="cuda")
torch.onnx.export(face_model, dummy, "face-specialist.onnx", input_names=["pixel_values"],
                  output_names=["logits"], opset_version=17, dynamo=False, external_data=False)
print("FACE_ONNX_EXPORTED", flush=True)
onnx_options = ort.SessionOptions()
onnx_options.intra_op_num_threads = 2
face_onnx = ort.InferenceSession("face-specialist.onnx", sess_options=onnx_options, providers=["CPUExecutionProvider"])
for row, (_, image, _, _) in zip(specialist_records, samples):
    logits = face_onnx.run(None, {"pixel_values":prepare_face(image)})[0].reshape(-1)
    p = np.exp(logits-logits.max()); score=float(p[1]/p.sum())
    row["onnx_score"] = score
    row["onnx_absolute_error"] = abs(score-row["reconstructed_score"])
    assert row["onnx_absolute_error"] < 1e-4, row
specialist_labels = np.array([row["label_fake"] for row in specialist_records])
specialist_scores = np.array([row["onnx_score"] for row in specialist_records])
model_digest = hashlib.sha256(Path("face-specialist.onnx").read_bytes()).hexdigest()
specialist_report = {"status":"passed", "gpu":gpu, "kind":"specialist_reconstruction_export_and_tiny_audit",
                    "utc":datetime.now(timezone.utc).isoformat(), "source_model":FACE_MODEL_ID,
                    "source_revision":FACE_REVISION, "source_sha256":FACE_SHA, "reference_precision":"Published trace replayed in FP32, promoting explicit BF16 casts to FP32", "onnx_sha256":model_digest,
                    "torch":torch.__version__, "onnxruntime":ort.__version__, "n_images":len(samples),
                    "sample_source":REPO, "sample_commit":COMMIT, "n_groups":8,
                    "accuracy_at_0_5":float(((specialist_scores>=.5)==specialist_labels).mean()),
                    "fake_detected_at_0_5":int(((specialist_scores>=.5)&(specialist_labels==1)).sum()),
                    "false_positive_at_0_5":int(((specialist_scores>=.5)&(specialist_labels==0)).sum()),
                    "limitations":"32 correlated example crops, FF++ may overlap training. This is a regression audit, not independent performance or calibration evidence. YuNet alignment on arbitrary uploads is an approximation of original dlib preprocessing and requires separate validation.",
                    "records":specialist_records}
Path("authenfy-specialist-validation.json").write_text(json.dumps(specialist_report, indent=2))
print("AUTHENFY_SPECIALIST_VALIDATION", json.dumps({k:v for k,v in specialist_report.items() if k!='records'}), flush=True)
files.download("authenfy-specialist-validation.json")
files.download("face-specialist.onnx")
